# Tome 10, Chapter 2: Laws of large numbers (Python companion)

Exercises first, solutions at the end. Monte Carlo integration and its error, the St Petersburg game played a million times, the recurrence of centred walks (Chung–Fuchs), the renewal–reward theorem and the inspection paradox, and a classic streaming algorithm whose correctness is a probability computation and whose usefulness is a law of large numbers: **reservoir sampling**. Requires `numpy`.

In [ ]:
import math
import numpy as np
rng = np.random.default_rng(20)

## Exercises

**P2.1 (★) Monte Carlo integration.** Estimate $I=\int_0^1e^{-x^2}\dd x$ by $\frac1n\sum f(U_k)$ with $U_k$ uniform. For $n=10^2,\dots,10^7$, compare the error with $\sigma/\sqrt n$, where $\sigma^2=\operatorname{Var}f(U)$. The strong law says the error tends to $0$; what does it not say?

**P2.2 (★) The St Petersburg game.** Simulate $1000$ players, each playing $n$ games, for $n=10^2,10^4,10^6$ (draw the number of tosses as a geometric variable). Report the quantiles of $S_n/(n\log_2n)$. Is the distribution concentrating around $1$ (Theorem 2.2)? How fast? What fraction of a player's total comes from the single best game?

**P2.3 (★★) Chung–Fuchs.** Walks on $\Z$ with steps $-1$ (probability $\frac23$) and $+2$ (probability $\frac13$) are centred. Count the visits to $0$ up to time $N$ for $N=10^3,\dots,10^6$, and compare with $\sqrt N$. Then add a drift: steps $-1$ and $+2$ with probabilities $0.67$ and $0.33$. What happens to the count?

**P2.4 (★★) Renewal–reward and the inspection paradox.** Light bulbs have i.i.d.\ lifetimes, uniform on $[0,2]$ (mean $1$). (a) Check $N(t)/t\to1$. (b) A visitor arrives at time $t=10^5$. Estimate the mean lifetime of the bulb burning at that moment, by simulation, and compare with $\mathbb E\xi^2/\mathbb E\xi=\frac43$. (c) Explain the answer with the renewal–reward theorem: over a long time, the fraction of time covered by bulbs of lifetime in $[a,b]$ is $\mathbb E[\xi\mathbf 1_{a\le\xi\le b}]/\mathbb E\xi$.

**P2.5 (★★★) CS application: reservoir sampling.** A stream of $n$ items arrives one at a time, $n$ unknown in advance; we must keep a uniform random sample of $k$ of them in memory $k$. *Algorithm R* (Waterman; analysed by Vitter, 1985): keep the first $k$; when item $i>k$ arrives, draw $J$ uniform in $\{1,\dots,i\}$ and, if $J\le k$, replace the $J$-th kept item by item $i$. (a) Prove by induction that after $i$ items, every $k$-subset of them is equally likely to be the reservoir. (b) Check the inclusion frequencies empirically for $n=20$, $k=5$ over $10^5$ runs. (c) Use a reservoir of size $k=1000$ over a stream of $10^6$ Pareto values ($\alpha=1.5$) to estimate the stream mean; compare the error with that of the full running mean.

## Solutions

### P2.1
**How to find it.** The exact value is $\frac{\sqrt\pi}2\operatorname{erf}(1)$. The variance of $f(U)$ is computable by the same Monte Carlo, or exactly.

In [ ]:
I = math.sqrt(math.pi) / 2 * math.erf(1)
sigma = math.sqrt(math.sqrt(math.pi / 2) / 2 * math.erf(math.sqrt(2)) - I**2)
for n in [10**k for k in range(2, 8)]:
    est = np.mean(np.exp(-rng.random(n)**2))
    print(f"n=1e{int(math.log10(n))}: error {abs(est - I):.2e}   sigma/sqrt(n) = {sigma / math.sqrt(n):.2e}")

The error follows $\sigma/\sqrt n$ (times a random factor of order $1$). The strong law gives convergence but no rate; the rate is the central limit theorem's (Chapter 4), and Berry–Esseen will say how far it can be trusted for a given $n$.

### P2.2
**How to find it.** A game pays $2^G$ with $G$ geometric on $\{1,2,\dots\}$ with parameter $\frac12$. For $n=10^6$ and $1000$ players we need $10^9$ games: draw them in chunks.

In [ ]:
def play(n, players=1000, chunk=10**5):
    tot = np.zeros(players); best = np.zeros(players)
    done = 0
    while done < n:
        m = min(chunk, n - done)
        for p in range(players):
            X = 2.0**rng.geometric(0.5, m)
            tot[p] += X.sum(); best[p] = max(best[p], X.max())
        done += m
    return tot, best

for n in (10**2, 10**4, 10**6):
    tot, best = play(n, players=1000 if n < 10**6 else 200)
    r = tot / (n * math.log2(n))
    q = np.quantile(r, [0.1, 0.5, 0.9])
    print(f"n=1e{int(math.log10(n))}: quantiles of S_n/(n log2 n) (10%,50%,90%) = {np.round(q, 3)},  median share of best game = {np.median(best / tot):.2f}")

The quantiles close in on $1$, but slowly: the corrections are of relative size $\frac{\log\log n}{\log n}$, and the upper quantiles stay inflated by the occasional huge prize. The best single game typically carries a fixed fraction of the total, around $\frac1{\log_2n}$ in order of magnitude but with a heavy tail: this is the signature of infinite mean.

### P2.3
**How to find it.** Visits to $0$ are the zeros of the cumulative sum. By the local limit theorem the walk is at $0$ at time $n$ with probability about $\frac1{\sqrt{2\pi\sigma^2n}}$, $\sigma^2=2$, so the expected number of visits by time $N$ is about $\sqrt{N/\pi}$.

In [ ]:
for probs, name in (((2/3, 1/3), "centred"), ((0.67, 0.33), "drift -0.01")):
    counts = np.zeros(4)
    for _ in range(40):
        S = np.cumsum(rng.choice([-1, 2], size=10**6, p=probs))
        counts += [np.sum(S[:10**k] == 0) for k in range(3, 7)]
    print(f"{name:12s} mean visits to 0 by N = 1e3..1e6 (40 walks):", np.round(counts / 40, 1))
print("sqrt(N/pi):", [round(math.sqrt(10**k / math.pi), 1) for k in range(3, 7)])

The centred walk keeps returning, on average about $\sqrt{N/\pi}$ times by time $N$; a single walk fluctuates a lot, since the number of returns of a recurrent walk is itself random of order $\sqrt N$. The tiny drift $-0.01$ makes the walk transient: after some tens of thousands of steps (when $0.01n$ outgrows the spread $\sqrt{2n}$) it leaves for good, and the count freezes.

### P2.4
**How to find it.** Simulate $\sum\xi_k$ until it exceeds $t$; the bulb burning at time $t$ is the one whose interval contains $t$. Repeat to average.

In [ ]:
t = 1e5
T = np.cumsum(rng.uniform(0, 2, int(1.2 * t)))
print("N(t)/t =", np.searchsorted(T, t) / t)
K = 20000
xi = rng.uniform(0, 2, (K, 200))
T = np.cumsum(xi, axis=1) + t - 150     # each row starts well before t; stationarity sets in fast
k = (T < t).sum(axis=1)                 # index of the bulb burning at time t
covering = xi[np.arange(K), k]
print("mean lifetime of the bulb burning at time t:", np.mean(covering), "  E xi^2 / E xi =", 4 / 3)

The bulb you find burning has mean lifetime $\frac43$, not $1$: long-lived bulbs cover more time, so a random time is more likely to fall in one of them. In renewal–reward form, the reward of a cycle is its length times $\mathbf 1_{\xi\in[a,b]}$, so the long-run fraction of time in such cycles is $\mathbb E[\xi\mathbf 1_{\xi\in[a,b]}]/\mathbb E\xi$: the lifetime seen at a random time has the size-biased density $x f(x)/\mu$, with mean $\mathbb E\xi^2/\mu$. The same bias explains why buses seem always late and why the average class size seen by students exceeds the average class size.

### P2.5
**How to find it.** (a) Induction on $i\ge k$: suppose the reservoir $R_{i-1}$ is a uniform $k$-subset of the first $i-1$ items. At step $i$, with probability $1-\frac ki$ nothing changes, and with probability $\frac ki$ a uniformly chosen slot is overwritten by item $i$. A $k$-subset $A$ not containing $i$ is obtained only by keeping $R_{i-1}=A$: probability $\binom{i-1}k^{-1}\frac{i-k}i=\binom ik^{-1}$. A subset $A=B\cup\{i\}$ is obtained from $R_{i-1}=B\cup\{x\}$ for each of the $i-k$ items $x\notin B$, by overwriting the slot of $x$: probability $(i-k)\binom{i-1}k^{-1}\frac ki\cdot\frac1k=\binom ik^{-1}$. So $R_i$ is uniform.

In [ ]:
def reservoir(stream, k, rng):
    R = list(stream[:k])
    for i in range(k, len(stream)):
        j = rng.integers(0, i + 1)
        if j < k:
            R[j] = stream[i]
    return R

n, k, runs = 20, 5, 10**5
counts = np.zeros(n)
items = np.arange(n)
for _ in range(runs):
    counts[reservoir(items, k, rng)] += 1
print("inclusion frequencies (should all be k/n = 0.25):", np.round(counts / runs, 3))

stream = rng.random(10**6)**(-1 / 1.5)                    # Pareto(1.5), mean 3
R = np.array(reservoir(stream, 1000, rng))
print(f"reservoir mean {R.mean():.3f}, full running mean {stream.mean():.3f}, true mean 3")

The inclusion frequencies are all $0.25$ up to sampling error. The reservoir mean is a mean of $1000$ i.i.d. draws from the stream, so the law of large numbers applies to it, with error of the order of the stream's spread over $\sqrt{1000}$ (here worse, because the variance is infinite and the error decays like $k^{-1/3}$, Chapter 5). Reservoir sampling is used in databases and stream processing for exactly this: a uniform sample of fixed size, maintained in one pass.